In [1]:
import sys
import os
import gc
import pandas as pd
import joblib
import matplotlib.pyplot as plt

#sistemare il warning di LightGBM fa creare warning a tutti gli altri modelli addestrati senza
#feature name, quindi meglio ingorarli
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Add the package root to sys.path so we can import stellar_classification
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'stellar_classification')))

import stellar_classification as sc

# Enable garbage collection
gc.enable()
gc.collect()

/home/nikko/miniconda3/envs/comp_phys/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


20

In [2]:
# Load dataset
data_path = '../stellar_classification/stellar_classification/data/star_classification.csv'
star = pd.read_csv(data_path)

print("First few rows:")
display(star.head())

print("\nData Info:")
star.info()

print("\nNull Values:")
print(star.isnull().sum())

print("\nClass Distribution:")
print(star["class"].value_counts(normalize=True) * 100)

# -------- Color indexes ----------

star["u_g"] = star["u"] - star["g"]
star["g_r"] = star["g"] - star["r"]
star["r_i"] = star["r"] - star["i"]
star["i_z"] = star["i"] - star["z"]

star.info()


First few rows:


,obj_ID,alpha,delta,u,g,r,i,z,run_ID,rerun_ID,cam_col,field_ID,spec_obj_ID,class,redshift,plate,MJD,fiber_ID
0,1.237661e+18,135.689107,32.494632,23.87882,22.27530,20.39501,19.16573,18.79371,3606,301,2,79,6.543777e+18,GALAXY,0.634794,5812,56354,171
1,1.237665e+18,144.826101,31.274185,24.77759,22.83188,22.58444,21.16812,21.61427,4518,301,5,119,1.176014e+19,GALAXY,0.779136,10445,58158,427
2,1.237661e+18,142.188790,35.582444,25.26307,22.66389,20.60976,19.34857,18.94827,3606,301,2,120,5.152200e+18,GALAXY,0.644195,4576,55592,299
3,1.237663e+18,338.741038,-0.402828,22.13682,23.77656,21.61162,20.50454,19.25010,4192,301,3,214,1.030107e+19,GALAXY,0.932346,9149,58039,775
4,1.237680e+18,345.282593,21.183866,19.43718,17.58028,16.49747,15.97711,15.54461,8102,301,3,137,6.891865e+18,GALAXY,0.116123,6121,56187,842



Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 18 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   obj_ID       100000 non-null  float64
 1   alpha        100000 non-null  float64
 2   delta        100000 non-null  float64
 3   u            100000 non-null  float64
 4   g            100000 non-null  float64
 5   r            100000 non-null  float64
 6   i            100000 non-null  float64
 7   z            100000 non-null  float64
 8   run_ID       100000 non-null  int64  
 9   rerun_ID     100000 non-null  int64  
 10  cam_col      100000 non-null  int64  
 11  field_ID     100000 non-null  int64  
 12  spec_obj_ID  100000 non-null  float64
 13  class        100000 non-null  str    
 14  redshift     100000 non-null  float64
 15  plate        100000 non-null  int64  
 16  MJD          100000 non-null  int64  
 17  fiber_ID     100000 non-null  int64  
dtypes: float64(10), int64(7)

In [3]:
# Preprocessing: Apply outlier removal, splits, standardization, and SMOTE
X_train, X_val, X_test, y_train, y_val, y_test, label_encoder, scaler, feature_names = sc.prepare_splits(
    star, 
    target_col='class', 
    test_size=0.2, 
    val_ratio=0.25, 
    random_state=42, 
    apply_outlier_removal=True
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")

print("\nClass Distribution:")
print(star["class"].value_counts(normalize=True) * 100)


Outliers removed: 8,715 rows  (91,285 remain)
X_train shape: (97032, 13)
X_val shape: (18257, 13)
X_test shape: (18257, 13)

Class Distribution:
class
GALAXY    59.445
STAR      21.594
QSO       18.961
Name: proportion, dtype: float64


In [5]:

models = sc.stack_training_models(
    X_train,
    y_train
)

stacking_clf = sc.make_stacking_classifier(
    models=models,
    n_jobs=2,
)

from sklearn.model_selection import GridSearchCV

param_grid = {
    'final_estimator__C': [0.1, 1, 10],
    'passthrough': [True, False],
    'cv' : [3, 5, 10] 
}

grid = GridSearchCV(
    estimator=stacking_clf,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=3,
    verbose=2,
    n_jobs=2,
    return_train_score=True
)
grid.fit(X_train, y_train)


Linear SVC trained.
Decision Tree trained.
Random Forest trained.
CatBoost trained.
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 97032, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (1.48 MB) transferred to GPU in 0.002299 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
LightGBM trained.
Fitting 3 folds for each of 18 candidates, totalling 54 fits


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 64688, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (0.99 MB) transferred to GPU in 0.002620 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.098597
[LightGBM] [Info] Start training from score -1.098597
[LightGBM] [Info] Start training from score -1.098643
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 64688, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling Op

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 43125, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (0.66 MB) transferred to GPU in 0.001421 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 43125, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling Op

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 64688, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (0.99 MB) transferred to GPU in 0.002361 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.098597
[LightGBM] [Info] Start training from score -1.098643
[LightGBM] [Info] Start training from score -1.098597
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 64688, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling Op

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 43125, number of used features: 13
[CV] END .....cv=3, final_estimator__C=0.1, passthrough=True; total time= 1.4min
[CV] END .....cv=3, final_estimator__C=0.1, passthrough=True; total time= 1.4min
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 43125, number of used features: 13


...

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 43125, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (0.66 MB) transferred to GPU in 0.001592 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 43125, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling Op

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 43125, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (0.66 MB) transferred to GPU in 0.001536 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 43125, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling Op

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 51750, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 13 dense feature groups (0.79 MB) transferred to GPU in 0.001723 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 51750, number of used features: 13
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling Op

Application terminated with error: ??+0 (0x7F2D7659CC1A)
??+0 (0x7F2D75E18446)
??+0 (0x7F2D77514414)
??+0 (0x7F2D77513431)
??+0 (0x7F2D77514638)
??+0 (0x7F2D76078E1E)
??+0 (0x7F2D76078C5A)
??+0 (0x7F2E674ABD58)
??+0 (0x7F2E6754A5DC)

(TCudaException) catboost/cuda/cuda_lib/cuda_base.h:245: CUDA error 2: out of memory
Terminating due to uncaught exception 0x2f9725b0410    what() -> "catboost/cuda/cuda_lib/cuda_base.h:245: CUDA error 2: out of memory"
 of type TCudaException
Fatal Python error: Aborted

Thread 0x00007f2d5effd6c0 (most recent call first):
  File "/home/nikko/miniconda3/envs/comp_phys/lib/python3.13/multiprocessing/pool.py", line 579 in _handle_results
  File "/home/nikko/miniconda3/envs/comp_phys/lib/python3.13/threading.py", line 995 in run
  File "/home/nikko/miniconda3/envs/comp_phys/lib/python3.13/threading.py", line 1044 in _bootstrap_inner
  File "/home/nikko/miniconda3/envs/comp_phys/lib/python3.13/threading.py", line 1015 in _bootstrap

Thread 0x00007f2d5f7fe6c0 (

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGABRT(-6)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.